# Building a **ReAct** Agent from Scratch

The *ReAct* pattern (Reasoning + Acting) lets a language model (LLM) do more than just answer questions. It can, for example, search for information on Wikipedia or perform calculations. In this pattern, we teach the model to ask for these actions to be run, then feed the results of those actions back in to improve its answers.

[ReAct](https://react-lm.github.io/) lets a language model run external actions, such as searches or calculations, and incorporate the results to improve its responses.

### LLM model
This notebook uses the *OpenAI* API and the *gpt-4o-mini* model.

In [ ]:
# Use your own OpenAI API key or create one at:
# https://platform.openai.com/account/api-keys

import getpass
key = getpass.getpass("Enter your OpenAI API key: ")

In [ ]:
# Imports
import re
import requests
from openai import OpenAI
from sympy import sympify
from typing import Dict, List, Optional

# OpenAI
client = OpenAI()

In [ ]:
# Testing the gpt-4o-mini model

model = "gpt-4o-mini"
prompt = "Hello, ChatGPT!"

chat_completion = client.chat.completions.create(
    model=model,
    messages=[{'role':'user', 'content': prompt}]
)

print(chat_completion.choices[0].message.content)

### 01 - Creating the Agent class

In [ ]:
class Agent:
    """
    Agent class that manages interactions with a language model.
    Initializes the system message context, appends user input,
    and calls the model to generate responses.

    Attributes:
        system (str): Initial system message. If provided, it's added to the message list.
        messages (List[Dict[str, str]]): History of messages exchanged between the user, system, and the model.

    Methods:
        __call__(prompt: str) -> str:
            Takes a user prompt, appends it to the message history, runs the model, and returns the response.

        execute(model: str = "gpt-4o-mini", temperature: float = 0) -> str:
            Runs the language model with the current message history and returns the response.
    """
    def __init__(self, system: str = "") -> None:
        self.system: str = system
        self.messages: List[Dict[str, str]] = []

        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, prompt: str) -> str:
        self.messages.append({"role": "user", "content": prompt})
        result: str = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self, model: str = "gpt-4o-mini", temperature: float = 0) -> str:
        completion = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=self.messages
        )
        return completion.choices[0].message.content

### 02 - Creating the **ReAct** prompt
*THOUGHT, ACTION, PAUSE, OBSERVATION*

In [ ]:
prompt = '''
You operate in a loop of THOUGHT, ACTION, PAUSE, OBSERVATION.
At the end of the loop, you produce an ANSWER.

Use THOUGHT to describe your reasoning about the question you've been asked.
Use ACTION to run one of the actions available to you - then return PAUSE.
OBSERVATION will be the result of running that action.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python, so make sure to use floating point syntax if needed.

get_cost:
e.g. get_cost: keyboard
Returns the cost of a keyboard.

get_current_weather:
e.g. get_current_weather: Brasília
Returns the current temperature of a city.

wikipedia:
e.g. wikipedia: LangChain
Returns a summary from a Wikipedia search.

Always look things up on Wikipedia if you have the opportunity to do so.

Example session #1:

Question: How much does a monitor cost?
THOUGHT: I should check the cost of a monitor using get_cost.
ACTION: get_cost: monitor
PAUSE

You will be called again with this:

OBSERVATION: A monitor costs $159.00.

You then produce the answer:

ANSWER: A monitor costs $159.00.


Example session #2:

Question: What is the capital of France?
THOUGHT: I should look up France on Wikipedia.
ACTION: wikipedia: France
PAUSE

You will be called again with this:

OBSERVATION: France is a country. The capital is Paris.

You then produce the answer:

ANSWER: The capital of France is Paris.


Example session #3:

Question: What's the weather like in São Paulo?
THOUGHT: I should get the current temperature for São Paulo using get_current_weather.
ACTION: get_current_weather: São Paulo
PAUSE

You will be called again with this:

OBSERVATION: 21°C

You then produce the answer:

ANSWER: The current temperature in São Paulo is 21°C.

'''.strip()

### 03 - Creating the tools
With tools (plain Python functions), we can make API calls and run our own code, expanding the capabilities of the environment the model was originally designed for.

In [ ]:
def calculate(expression: str) -> float:
    """
    Evaluates a mathematical expression given as a string and returns the result
    rounded to at most two decimal places. Uses the SymPy library.

    Args:
        expression (str): The mathematical expression to evaluate, e.g. "(2 * 10) + (3 * 15)".

    Returns:
        float: The evaluated result, rounded to at most two decimal places.
    """
    try:
        result = sympify(expression).evalf()
        return round(float(result), 2)
    except Exception as e:
        return f"Error: {e}"


def get_cost(item: str) -> str:
    """
    Simulates a call to a fictitious store API.
    Returns the cost of a tech item.

    Args:
        item (str): The name of the tech item.

    Returns:
        str: A message with the item's price, or a generic message for other items.
    """
    if item == 'mouse':
        return 'A mouse costs $19.90'
    elif item == 'keyboard':
        return 'A keyboard costs $29.90'
    elif item == 'monitor':
        return 'A monitor costs $159.00'
    else:
        return 'Other items cost $39.00.'


def get_current_weather(city: str) -> Optional[str]:
    """
    Gets the current temperature for a city using the wttr.in API.

    Args:
        city (str): The name of the city to get the weather for.

    Returns:
        Optional[str]: A formatted string with the current temperature in Celsius.
        Returns `None` if there's an error in the request or the data.
    """
    base_url = f"http://wttr.in/{city}?format=j1"
    response = requests.get(base_url)

    if response.status_code != 200:
        return None

    data = response.json()

    try:
        temperature = data['current_condition'][0]['temp_C']
    except (KeyError, IndexError):
        return None

    return f"{temperature}°C"


def wikipedia(search_term: str) -> Optional[str]:
    """
    Queries the Wikipedia API and returns the snippet from the first result found.

    Args:
        search_term (str): The search term to query on Wikipedia.

    Returns:
        Optional[str]: The snippet from the first result found. Returns `None` if there are no results.
    """
    response = requests.get('https://en.wikipedia.org/w/api.php', params={
        'action': 'query',
        'list': 'search',
        'srsearch': search_term,
        'format': 'json'
    })
    results = response.json().get('query').get('search', [])

    if not results:
        return None
    return results[0]['snippet']

In [ ]:
# dictionary mapping tool names to the functions themselves
tools = {
    'calculate': calculate,
    'get_cost': get_cost,
    'get_current_weather': get_current_weather,
    'wikipedia': wikipedia,
}

### 04 - Testing the Agent - Step by Step

#### Example 1

In [ ]:
# creating an Agent instance
agent = Agent(prompt)

result = agent('How much does a mouse cost?')
print(result)

In [ ]:
# creating the next prompt, which will be used as an Observation and sent to the language model
next_prompt = f"OBSERVATION: {get_cost('mouse')}"
print(next_prompt)

In [ ]:
# Agent's final answer
print(agent(next_prompt))

In [ ]:
# Agent's message history
agent.messages[1:]

#### Example 2

In [ ]:
agent = Agent(prompt)

question = "I want to buy a keyboard and a mouse. How much do they cost in total?"
print(agent(question))

In [ ]:
next_prompt = f'OBSERVATION: {get_cost("keyboard")}'
print(next_prompt)

In [ ]:
# back to the Agent after the first observation
print(agent(next_prompt))

In [ ]:
next_prompt = f'OBSERVATION: {get_cost("mouse")}'
print(next_prompt)

In [ ]:
# back to the Agent after the second observation
print(agent(next_prompt))

In [ ]:
next_prompt = f'OBSERVATION: {calculate("29.90 + 19.90")}'
print(next_prompt)

In [ ]:
# Agent's final answer
print(agent(next_prompt))

#### Example 3

In [ ]:
agent = Agent(prompt)

question = "LangChain"
print(agent(question))

In [ ]:
# calling the wikipedia() function and creating the next prompt
next_prompt = f'OBSERVATION: {wikipedia(question)}'
print(next_prompt)

In [ ]:
# Agent's final answer
print(agent(next_prompt))

In [ ]:
# Agent's message history
agent.messages[1:]

#### Example 4

In [ ]:
agent = Agent(prompt)

question = "What's the weather like in Santos?"
print(agent(question))

In [ ]:
next_prompt = f'OBSERVATION: {get_current_weather("Santos")}'
print(next_prompt)

In [ ]:
# Agent's final answer
print(agent(next_prompt))

In [ ]:
# Agent's message history
agent.messages[1:]

-----

### Automating the Agent loop

In [ ]:
# regex to find the 'ACTION' string
action_re = re.compile(r'^ACTION: (\w+): (.*)$')

In [ ]:
def call_agent(question: str, max_turns: int = 5) -> str:
    i: int = 0
    bot: Agent = Agent(prompt)
    next_prompt: str = question

    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)

         # using the regex to parse the Agent's response
        actions = [
            action_re.match(a) for a in result.split('\n') if action_re.match(a)
        ]

        if actions:
            action, action_input = actions[0].groups()

            if action not in tools:
                raise Exception(f'Unknown action: {action}: {action_input}')

            print(f' -- running --> {action} {action_input}')
            observation = tools[action](action_input)

            print(f'OBSERVATION: {observation}')
            next_prompt = f'OBSERVATION: {observation}'
        else:
            return

In [ ]:
question = "I want to buy 2 monitors and 2 mice. How much do I have to pay?"
call_agent(question)

In [ ]:
question = "Andrew Ng"
call_agent(question)

In [ ]:
question = "What's the weather like in Lisbon right now?"
call_agent(question)